# Exercises — Chapters 6 and 7
*Elmasri & Navathe, 7th edition — COMPANY database.*

Ten exercises on the COMPANY database (Figure 5.6), from easy to hard: ★☆☆ easy (Chapter 6), ★★☆ medium and ★★★ hard (Chapter 7). Each exercise gives the query in English and the columns the result must have.

**Checking an answer:** every SQL box starts with `%%pg --check N`. After the query runs, the box compares its result with the expected answer of exercise N and prints **✔ correct** or **✘** with what is wrong (number of columns, number of rows, values, or order). Column names do not matter, numbers are compared to two decimals, and the row order matters only when the exercise asks for it. The expected answers are stored only as fingerprints (hashes), so they cannot be read from the notebook.

The notebook is self-contained: run the two setup cells first (in Colab they install PostgreSQL into the runtime, about a minute), or use **Runtime → Run all**.

## 0. Setup

In [ ]:
#@title Helper code: starts PostgreSQL and defines the %%pg and %show commands (run this cell; no need to read it) { display-mode: "form" }
# Self-contained helper for the Elmasri & Navathe (7th ed.) Chapter 6-7 SQL notebooks.
# Nothing has to be uploaded: the COMPANY schema and the Figure 5.6 data are embedded below.
#
# Where PostgreSQL comes from (chosen automatically by sqlnb.setup()):
#   * Google Colab          -> installed into the Colab runtime with apt-get (once per runtime)
#   * PyCharm / Jupyter     -> a Docker container "elmasri-company-pg" (port 55432), Docker Desktop on macOS
#   * any Linux VM as root  -> apt-get, when Docker is not available
#   * SQLNB_DSN is set      -> that PostgreSQL server, e.g. "host=localhost port=5432 user=postgres password=... dbname=company"
#
# Commands defined for the notebook:
#   %%pg [--expect-error]   run every SQL statement in the cell; results are printed as tables
#   %%pg --check N          exercises: also compare the cell's last result with the expected answer of exercise N
#   %pg SELECT ...          one-line SQL
#   %show [TABLE ...]       print tables (no argument: the six COMPANY tables)
#   sqlnb.reset_company()   restore COMPANY to Figure 6.1/6.2 + the Figure 5.6 database state
# (The cell command is %%pg, not %%sql: PyCharm turns %%sql cells into its own DuckDB-based SQL cells.)
import datetime as _dt
import hashlib as _hashlib
import html as _html
import importlib as _importlib
import json as _json
import os as _os
import re as _re
import shutil as _shutil
import socket as _socket
import subprocess as _subprocess
import sys as _sys
import time as _time
import types as _types
from decimal import Decimal as _Decimal

_CFG = {
    "image": _os.environ.get("SQLNB_PG_IMAGE", "postgres:16"),
    "container": _os.environ.get("SQLNB_CONTAINER", "elmasri-company-pg"),
    "host": "127.0.0.1",
    "port": int(_os.environ.get("SQLNB_PORT", "55432")),
    "user": "postgres",
    "password": "postgres",
    "db": "company",
    "search_path": "company, public",
}
_COMPANY_TABLES = ["EMPLOYEE", "DEPARTMENT", "DEPT_LOCATIONS", "PROJECT", "WORKS_ON", "DEPENDENT"]
if "_ST" not in globals():  # connection state survives re-running this cell
    _ST = {"conn": None, "dsn": None, "mode": None, "notices": []}


class SetupError(RuntimeError):
    """A setup problem, reported with instructions."""


class SQLCellError(RuntimeError):
    """Unexpected SQL result in strict (test) mode."""


# ---------------------------------------------------------------- output
class _Out:
    def __init__(self, text, html_):
        self.text, self.html_ = text, html_

    def __repr__(self):
        return self.text

    def _repr_html_(self):
        return self.html_


def _display(obj):
    try:
        from IPython.display import display
        display(obj)
    except Exception:
        print(repr(obj))


_COLORS = {"info": ("#1b5e20", "#e8f5e9"), "muted": ("#888888", None), "warn": ("#7a4b00", "#fff4e0"),
           "error": ("#8b0000", "#fdecea"), "expected": ("#7a4b00", "#fff4e0")}


def _msg(text, kind="info"):
    fg, bg = _COLORS.get(kind, _COLORS["info"])
    body = _html.escape(text).replace("\n", "<br>")
    box = f"background:{bg};padding:4px 8px;border-radius:4px;" if bg else ""
    _display(_Out(text, f'<div style="font-family:monospace;font-size:12px;color:{fg};{box}margin:2px 0">{body}</div>'))


def _fmt(v):
    if v is None:
        return "NULL"
    if isinstance(v, bool):
        return "TRUE" if v else "FALSE"
    if isinstance(v, (_dt.date, _dt.time)) and not isinstance(v, _dt.datetime):
        return v.isoformat()
    return str(v)


def _is_num(v):
    return isinstance(v, (int, float, _Decimal)) and not isinstance(v, bool)


def _table_text(cols, rows):
    cells = [[_fmt(v) for v in r] for r in rows]
    idx = [str(i) for i in range(1, len(rows) + 1)]
    iw = max([len(s) for s in idx] + [1])
    widths = [max([len(c)] + [len(r[j]) for r in cells]) for j, c in enumerate(cols)]
    right = [bool(rows) and all(_is_num(r[j]) or r[j] is None for r in rows) for j in range(len(cols))]

    def line(first, vals):
        parts = [(v.rjust(w) if right[j] else v.ljust(w)) for j, (v, w) in enumerate(zip(vals, widths))]
        return first.rjust(iw) + "  " + "  ".join(parts)

    out = [line("", list(cols))] + [line(i, r) for i, r in zip(idx, cells)]
    return "\n".join(s.rstrip() for s in out)


def _table_html(cols, rows):
    bd = "border:1px solid rgba(128,128,128,.35);padding:2px 8px;"
    head = "".join(f'<th style="{bd}background:rgba(128,128,128,.12);text-align:left">{_html.escape(str(c))}</th>'
                   for c in cols)
    body = []
    for i, r in enumerate(rows, 1):
        tds = [f'<td style="{bd}color:#888;text-align:right">{i}</td>']
        for v in r:
            align = "right" if _is_num(v) else "left"
            txt = ('<span style="color:#999;font-style:italic">NULL</span>' if v is None
                   else _html.escape(_fmt(v)))
            tds.append(f'<td style="{bd}text-align:{align};white-space:pre">{txt}</td>')
        body.append("<tr>" + "".join(tds) + "</tr>")
    return (f'<table style="border-collapse:collapse;font-size:13px;margin:2px 0 10px 0">'
            f'<thead><tr><th style="{bd}background:rgba(128,128,128,.12)"></th>{head}</tr></thead>'
            f'<tbody>{"".join(body)}</tbody></table>')


def _show_result(cols, rows, caption=None):
    if len(cols) == 1 and cols[0] == "QUERY PLAN":  # EXPLAIN output: keep the indentation
        plan = "\n".join(str(r[0]) for r in rows)
        if caption:
            _msg(f"> {caption}", "muted")
        _display(_Out(plan, '<pre style="font-size:12px;margin:2px 0 8px 0;padding:6px 8px;'
                            f'background:rgba(128,128,128,.10);border-radius:4px">{_html.escape(plan)}</pre>'))
        return
    label = f"{len(rows)} row" + ("" if len(rows) == 1 else "s")
    _msg(f"> {caption}   ({label})" if caption else label, "muted")
    if rows:
        _display(_Out(_table_text(cols, rows), _table_html(cols, rows)))


# ---------------------------------------------------------------- python driver
def _ensure_psycopg():
    try:
        import psycopg  # noqa: F401
        return
    except ImportError:
        pass
    _msg("Installing the PostgreSQL driver for Python (pip install psycopg[binary]) ...")
    r = _subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "psycopg[binary]"],
                        capture_output=True, text=True)
    if r.returncode != 0:
        raise SetupError("Could not install psycopg:\n" + (r.stderr or r.stdout)[-1500:])
    _importlib.invalidate_caches()


_ensure_psycopg()
import psycopg  # noqa: E402
from psycopg.pq import TransactionStatus  # noqa: E402


def _run(cmd, timeout=600, env=None):
    try:
        return _subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, env=env)
    except FileNotFoundError as e:
        return _subprocess.CompletedProcess(cmd, 127, "", str(e))
    except _subprocess.TimeoutExpired:
        return _subprocess.CompletedProcess(cmd, 124, "", f"{' '.join(cmd)}: timed out")


def _port_open(port):
    with _socket.socket(_socket.AF_INET, _socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((_CFG["host"], port)) == 0


# ---------------------------------------------------------------- PostgreSQL via apt-get (Colab)
def _in_colab():
    return "google.colab" in _sys.modules or "COLAB_RELEASE_TAG" in _os.environ


def _clusters():
    """[(version, name, port, status)] from pg_lsclusters (Debian/Ubuntu postgresql-common)."""
    r = _run(["pg_lsclusters", "-h"], timeout=30)
    out = []
    for line in r.stdout.splitlines():
        p = line.split()
        if len(p) >= 4 and p[2].isdigit():
            out.append((p[0], p[1], int(p[2]), p[3]))
    return out


def _as_postgres(args):
    """Run a command as the postgres OS user (works with or without sudo)."""
    last = None
    for prefix in (["runuser", "-u", "postgres", "--"], ["sudo", "-n", "-u", "postgres"]):
        r = _run(prefix + args, timeout=60)
        if r.returncode != 127:
            return r
        last = r
    return last


def _setup_apt():
    if _sys.platform != "linux" or not hasattr(_os, "geteuid") or _os.geteuid() != 0:
        raise SetupError("Installing PostgreSQL with apt-get needs a Linux machine and root (as in Colab).")
    if not _clusters():
        _msg("Installing PostgreSQL into this runtime with apt-get (once per runtime, about a minute) ...")
        env = dict(_os.environ, DEBIAN_FRONTEND="noninteractive")
        cmd = ["apt-get", "install", "-y", "-q", "postgresql"]
        r = _run(cmd, env=env)
        if r.returncode != 0:  # package lists missing or stale
            _run(["apt-get", "update", "-q"], env=env)
            r = _run(cmd, env=env)
        if r.returncode != 0:
            raise SetupError("apt-get install postgresql failed:\n" + (r.stderr or r.stdout)[-1500:])
        if not _clusters():  # the package normally creates the "main" cluster
            vers = sorted(_os.listdir("/usr/lib/postgresql"), key=lambda v: float(v))
            _run(["pg_createcluster", vers[-1], "main"], timeout=120)
    ver, name, port, status = sorted(_clusters(), key=lambda c: float(c[0]))[-1]
    if status != "online" or not _port_open(port):
        _msg(f"Starting PostgreSQL {ver} ...")
        r = _run(["pg_ctlcluster", ver, name, "start"], timeout=120)
        if r.returncode != 0 and not _port_open(port):
            r2 = _run(["service", "postgresql", "start"], timeout=120)
            if r2.returncode != 0 and not _port_open(port):
                raise SetupError("Could not start PostgreSQL:\n" + (r.stderr or r.stdout) + (r2.stderr or r2.stdout))
    # local (Unix-socket) connections of the postgres OS user need no password; set one for TCP
    r = _as_postgres(["psql", "-q", "-p", str(port), "-c",
                      f"ALTER USER {_CFG['user']} PASSWORD '{_CFG['password']}'"])
    if r is None or r.returncode != 0:
        raise SetupError("Could not set the postgres password:\n" + ((r.stderr or r.stdout) if r else ""))
    return port


# ---------------------------------------------------------------- PostgreSQL via Docker (PyCharm / Jupyter)
_DOCKER_CANDIDATES = ["/usr/local/bin/docker", "/opt/homebrew/bin/docker", "~/.docker/bin/docker",
                      "/Applications/Docker.app/Contents/Resources/bin/docker", "~/.orbstack/bin/docker",
                      "~/.rd/bin/docker", "/usr/bin/docker"]


def _find_docker():
    if _os.environ.get("SQLNB_DOCKER"):
        return _os.environ["SQLNB_DOCKER"]
    found = _shutil.which("docker")
    if found:
        return found
    for cand in _DOCKER_CANDIDATES:
        p = _os.path.expanduser(cand)
        if _os.path.isfile(p) and _os.access(p, _os.X_OK):
            return p
    return None


def _docker(*args, timeout=60):
    exe = _find_docker()
    if exe is None:
        return _subprocess.CompletedProcess(["docker", *args], 127, "", "docker not found")
    # GUI apps such as PyCharm start with a short PATH; Docker's credential helpers live next to docker
    extra = [_os.path.dirname(exe), "/usr/local/bin", "/opt/homebrew/bin", _os.path.expanduser("~/.docker/bin"),
             "/Applications/Docker.app/Contents/Resources/bin"]
    env = dict(_os.environ, PATH=_os.pathsep.join(extra + [_os.environ.get("PATH", "")]))
    return _run([exe, *args], timeout=timeout, env=env)


def _docker_daemon():
    r = _docker("info", "--format", "{{.ServerVersion}}", timeout=20)
    out = r.stdout.strip()
    return out if r.returncode == 0 and out and "error" not in out.lower() else None


def _setup_docker():
    ver = _docker_daemon()
    if not ver and _sys.platform == "darwin":
        _msg("Docker is not running; starting Docker Desktop (open -a Docker) ...")
        _run(["open", "-a", "Docker"], timeout=30)
        deadline = _time.time() + 120
        while not ver and _time.time() < deadline:
            _time.sleep(3)
            ver = _docker_daemon()
    if not ver:
        raise SetupError("The Docker daemon is not running.\n"
                         "  * macOS: open Docker Desktop, wait until it says it is running, then run this cell again.")
    _msg(f"Docker {ver} is running.")
    name = _CFG["container"]
    r = _docker("inspect", "-f", "{{.State.Status}}", name)
    status = r.stdout.strip() if r.returncode == 0 else None
    if status is None:
        if _docker("image", "inspect", _CFG["image"]).returncode != 0:
            _msg(f"Downloading the {_CFG['image']} image (first run only, about a minute) ...")
            p = _docker("pull", _CFG["image"], timeout=900)
            if p.returncode != 0:
                raise SetupError(f"Could not download {_CFG['image']}:\n{(p.stderr or p.stdout)[-800:]}")
        _msg(f"Creating container '{name}' ({_CFG['image']}) ...")
        err = ""
        for port in range(_CFG["port"], _CFG["port"] + 10):
            if _port_open(port):
                _msg(f"  port {port} is busy, trying the next one", "muted")
                continue
            r = _docker("run", "-d", "--name", name, "-e", f"POSTGRES_PASSWORD={_CFG['password']}",
                        "-e", f"POSTGRES_DB={_CFG['db']}", "-e", "TZ=Europe/Istanbul",
                        "-p", f"{_CFG['host']}:{port}:5432", "--label", "purpose=elmasri-sql-notebooks",
                        _CFG["image"], timeout=300)
            if r.returncode == 0:
                return port
            err = (r.stderr or r.stdout).strip()
            _docker("rm", "-f", name)
            if not any(k in err.lower() for k in ("port is already allocated", "address already in use", "bind")):
                break
        raise SetupError("Could not create the PostgreSQL container:\n" + err)
    if status == "running":
        _msg(f"Container '{name}' is already running; reusing it.")
    elif status == "paused":
        _docker("unpause", name)
    else:
        _msg(f"Container '{name}' is {status}; starting it ...")
        r = _docker("start", name, timeout=120)
        if r.returncode != 0:
            raise SetupError(f"Could not start the container:\n{(r.stderr or r.stdout).strip()}\n"
                             "  * To delete and recreate it: sqlnb.remove_container(); sqlnb.setup()")
    r = _docker("port", name, "5432/tcp")
    m = _re.search(r":(\d+)\s*$", r.stdout.strip().splitlines()[0]) if r.returncode == 0 and r.stdout.strip() else None
    if not m:
        raise SetupError(f"Container '{name}' does not publish port 5432. Recreate it: "
                         "sqlnb.remove_container(); sqlnb.setup()")
    return int(m.group(1))


# ---------------------------------------------------------------- connection
def _with_db(dsn, dbname):
    params = psycopg.conninfo.conninfo_to_dict(dsn)
    params["dbname"] = dbname
    return psycopg.conninfo.make_conninfo(**params)


def _wait_ready(dsn, timeout=120):
    deadline, announced, last = _time.time() + timeout, False, ""
    while _time.time() < deadline:
        try:
            with psycopg.connect(dsn, connect_timeout=3, autocommit=True) as c:
                c.execute("SELECT 1")
            return
        except psycopg.OperationalError as e:
            last = (str(e).strip().splitlines() or [repr(e)])[0]
            if not announced:
                _msg("Waiting for PostgreSQL to accept connections ...")
                announced = True
            _time.sleep(1)
    raise SetupError(f"PostgreSQL was not ready within {timeout} s. Last error: {last}")


def _on_notice(diag):
    text = f"{diag.severity or 'NOTICE'}: {diag.message_primary}"
    if diag.message_detail:  # e.g. the objects dropped by CASCADE
        text += "\n" + "\n".join("  " + s for s in diag.message_detail.splitlines())
    _ST["notices"].append(text)


def _connect(dsn=None):
    dsn = dsn or _ST["dsn"]
    if dsn is None:
        raise SetupError("Run the setup cell (sqlnb.setup()) first.")
    try:  # create the company database if it is missing
        with psycopg.connect(_with_db(dsn, "postgres"), autocommit=True) as c:
            if not c.execute("SELECT 1 FROM pg_database WHERE datname = %s", (_CFG["db"],)).fetchone():
                c.execute(f'CREATE DATABASE "{_CFG["db"]}"')
    except psycopg.OperationalError:
        pass
    target = _with_db(dsn, _CFG["db"])
    old = _ST["conn"]
    if old is not None and not old.closed:
        old.close()
    conn = psycopg.connect(target, autocommit=True)
    conn.execute(f"SET search_path TO {_CFG['search_path']}")
    try:
        conn.execute("SET TimeZone TO 'Europe/Istanbul'")
    except psycopg.Error:  # the server has no time-zone database
        conn.execute("SET TimeZone TO 'UTC'")
        _msg("Note: the server has no 'Europe/Istanbul' time zone; times are shown in UTC.", "muted")
    conn.add_notice_handler(_on_notice)
    try:  # other sessions (PyCharm's Database window, psql) look in COMPANY first too
        conn.execute(f"ALTER DATABASE {_CFG['db']} SET search_path TO {_CFG['search_path']}")
    except psycopg.Error:
        pass
    _ST["conn"], _ST["dsn"] = conn, target
    return conn


def _reconnect():
    try:
        conn = _connect()
    except psycopg.OperationalError:
        raise SetupError("The connection to PostgreSQL was lost and the server is not reachable\n"
                         "(the Colab runtime was reset, or Docker / the computer was restarted).\n"
                         "  * Run the setup cell (sqlnb.setup()) again.") from None
    _msg("The connection was lost; reconnected.", "warn")
    return conn


def _require_conn():
    conn = _ST["conn"]
    if conn is None:
        raise SetupError("Not connected: run the setup cell (sqlnb.setup()) first.")
    if conn.closed or conn.broken:
        conn = _reconnect()
    return conn


def _connection_info():
    if not _ST["dsn"]:
        return "(not connected)"
    p = psycopg.conninfo.conninfo_to_dict(_ST["dsn"])
    return f"postgresql://{p.get('user')}@{p.get('host')}:{p.get('port')}/{p.get('dbname')}  (password: {p.get('password', '')})"


def _setup(reset=False):
    """Start PostgreSQL (Colab: apt-get; elsewhere: Docker), connect, register %%pg / %pg / %show."""
    try:
        if _os.environ.get("SQLNB_DSN"):
            _msg("SQLNB_DSN is set: connecting to that PostgreSQL server.")
            dsn, mode = _os.environ["SQLNB_DSN"], "dsn"
            _wait_ready(_with_db(dsn, "postgres"), timeout=30)
        else:
            if _in_colab():
                mode = "apt"
            elif _find_docker() is not None:
                mode = "docker"
            elif _sys.platform == "linux" and _shutil.which("apt-get") and _os.geteuid() == 0:
                mode = "apt"
            else:
                raise SetupError(
                    "No way to start PostgreSQL was found.\n"
                    "  * Google Colab: just run this cell (PostgreSQL is installed automatically).\n"
                    "  * macOS / PyCharm: install Docker Desktop (https://www.docker.com/products/docker-desktop/), open it once.\n"
                    "  * Docker elsewhere: import os; os.environ['SQLNB_DOCKER'] = '/full/path/to/docker'\n"
                    "  * An existing server: os.environ['SQLNB_DSN'] = 'host=... port=... user=... password=... dbname=company'")
            port = _setup_apt() if mode == "apt" else _setup_docker()
            dsn = (f"host={_CFG['host']} port={port} user={_CFG['user']} "
                   f"password={_CFG['password']} dbname={_CFG['db']}")
            _wait_ready(_with_db(dsn, "postgres"))
        conn = _connect(dsn)
        _ST["mode"] = mode
    except SetupError as e:
        _msg("SETUP ERROR\n" + str(e), "error")
        raise
    _register_magics()
    server = conn.execute("SHOW server_version").fetchone()[0].split()[0]
    where = {"apt": "installed in this runtime", "docker": f"Docker container '{_CFG['container']}'",
             "dsn": "SQLNB_DSN"}[mode]
    _msg(f"PostgreSQL {server} is ready ({where}): {_connection_info()}")
    _msg("search_path = company, public   |   %%pg and %show are ready to use.")
    if reset:
        _reset_company()


def _stop():
    """Docker: stop the container (data kept). Colab: stop the server."""
    if _ST["conn"] is not None and not _ST["conn"].closed:
        _ST["conn"].close()
    if _ST["mode"] == "docker":
        r = _docker("stop", _CFG["container"], timeout=60)
        _msg("Container stopped." if r.returncode == 0 else (r.stderr or r.stdout).strip())
    elif _ST["mode"] == "apt":
        for ver, name, _port, _status in _clusters():
            _run(["pg_ctlcluster", ver, name, "stop"], timeout=60)
        _msg("PostgreSQL stopped.")


def _remove_container():
    """Docker only: delete the container (the next setup() creates a fresh one)."""
    if _ST["conn"] is not None and not _ST["conn"].closed:
        _ST["conn"].close()
    r = _docker("rm", "-f", _CFG["container"], timeout=60)
    _msg("Container removed." if r.returncode == 0 else (r.stderr or r.stdout).strip())


# ---------------------------------------------------------------- running SQL
_DOLLAR = _re.compile(r"\$([A-Za-z_][A-Za-z_0-9]*)?\$")


def _strip_comments(s):
    s = _re.sub(r"/\*.*?\*/", " ", s, flags=_re.S)
    return "\n".join(line.split("--", 1)[0] for line in s.splitlines())


def _split_sql(text):
    """Split at ';' outside quotes, $$ bodies and comments."""
    stmts, buf, i, n = [], [], 0, len(text)
    while i < n:
        c = text[i]
        if text.startswith("--", i):
            j = text.find("\n", i)
            j = n if j == -1 else j
        elif text.startswith("/*", i):
            depth, j = 1, i + 2
            while j < n and depth:
                if text.startswith("/*", j):
                    depth, j = depth + 1, j + 2
                elif text.startswith("*/", j):
                    depth, j = depth - 1, j + 2
                else:
                    j += 1
        elif c in ("'", '"'):
            j = i + 1
            while j < n:
                if text[j] == c:
                    if j + 1 < n and text[j + 1] == c:
                        j += 2
                        continue
                    break
                j += 1
            j += 1
        elif c == "$" and _DOLLAR.match(text, i) and not (i > 0 and (text[i - 1].isalnum() or text[i - 1] == "_")):
            m = _DOLLAR.match(text, i)
            j = text.find(m.group(0), m.end())
            j = n if j == -1 else j + len(m.group(0))
        elif c == ";":
            stmts.append("".join(buf))
            buf, i = [], i + 1
            continue
        else:
            j = i + 1
        buf.append(text[i:j])
        i = j
    stmts.append("".join(buf))
    return [s.strip() for s in stmts if _strip_comments(s).strip()]


def _header(text):
    for line in text.strip().splitlines():
        line = line.strip()
        if line:
            return line.lstrip("-").strip() if line.startswith("--") else ""
    return ""


def _caption(stmt):
    comments, code = [], []
    for line in stmt.splitlines():
        s = line.strip()
        if s.startswith("--") and not code:
            body = s.lstrip("-").strip()
            if not _re.match(r"(?i)^(slide|slides|textbook)\b", body):
                comments.append(body)
        elif s:
            code.append(s)
    if comments:
        return " ".join(comments)
    joined = " ".join(code)
    return joined[:90] + (" ..." if len(joined) > 90 else "")


def _jsonable(v):
    if v is None or isinstance(v, (bool, int, str, float)):
        return v
    if isinstance(v, (_dt.date, _dt.datetime, _dt.time)):
        return v.isoformat()
    return str(v)


def _record(entry):
    path = _os.environ.get("SQLNB_RECORD")
    if path:
        with open(path, "a", encoding="utf-8") as f:
            f.write(_json.dumps(entry, ensure_ascii=False) + "\n")


def _flush_notices():
    while _ST["notices"]:
        _msg(_ST["notices"].pop(0), "muted")


_ROW_KEYWORDS = ("SELECT", "WITH", "VALUES", "TABLE", "EXPLAIN", "SHOW")

# ---------------------------------------------------------------- exercise checker
# Expected answers are stored only as hashes of the normalized result, so they reveal nothing.
_EXPECTED = {1: {'h': '40054dc640fea4726df2', 'hu': 'c9939614aa206456f938', 'ncols': 3, 'nrows': 3, 'ordered': True},
 2: {'h': None, 'hu': 'c482a5f2479b508189f2', 'ncols': 2, 'nrows': 2, 'ordered': False},
 3: {'h': None, 'hu': 'b4dda64fd8998d36cdcf', 'ncols': 3, 'nrows': 3, 'ordered': False},
 4: {'h': None, 'hu': '185427b186101441f345', 'ncols': 4, 'nrows': 2, 'ordered': False},
 5: {'h': '7671838ef847439f82e5', 'hu': '7671838ef847439f82e5', 'ncols': 3, 'nrows': 4, 'ordered': True},
 6: {'h': '79cba3f179fbffd54ad4', 'hu': '4ae1504afb228bd30725', 'ncols': 3, 'nrows': 8, 'ordered': True},
 7: {'h': None, 'hu': 'a764b77895f9b5b219d2', 'ncols': 2, 'nrows': 2, 'ordered': False},
 8: {'h': None, 'hu': 'a701146dc032001d6254', 'ncols': 2, 'nrows': 1, 'ordered': False},
 9: {'h': None, 'hu': '7e16acc90278a4f29515', 'ncols': 4, 'nrows': 3, 'ordered': False},
 10: {'h': None, 'hu': 'ca9a97ed4e242e1f45b9', 'ncols': 2, 'nrows': 7, 'ordered': False}}


def _norm_val(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return "TRUE" if v else "FALSE"
    if isinstance(v, (int, float, _Decimal)):
        return str(_Decimal(str(v)).quantize(_Decimal("0.01")))  # 3, 3.0 and 3.000 compare equal
    if isinstance(v, (_dt.date, _dt.datetime, _dt.time)):
        return v.isoformat()
    return str(v).rstrip()  # CHAR(n) padding does not matter


def _result_hash(rows, ordered):
    norm = [[_norm_val(v) for v in r] for r in rows]
    if not ordered:
        norm.sort(key=lambda r: _json.dumps(r))
    return _hashlib.sha256(_json.dumps(norm).encode("utf-8")).hexdigest()[:20]


def _check(n, last, error, header=""):
    exp = _EXPECTED.get(n)
    if exp is None:
        _msg(f"There is no exercise {n} in this notebook.", "warn")
        return
    if error is not None:
        ok, reason, text = False, "error", f"Exercise {n}: fix the error above first."
    elif last is None:
        ok, reason, text = False, "empty", f"Exercise {n}: no result yet. Write a SELECT query in this cell and run it."
    else:
        cols, rows = last
        if len(cols) != exp["ncols"]:
            ok, reason = False, "ncols"
            text = f"Exercise {n}: not yet. The result should have {exp['ncols']} columns; yours has {len(cols)}."
        elif len(rows) != exp["nrows"]:
            ok, reason = False, "nrows"
            text = f"Exercise {n}: not yet. The result should have {exp['nrows']} rows; yours has {len(rows)}."
        elif _result_hash(rows, False) != exp["hu"]:
            ok, reason = False, "values"
            text = f"Exercise {n}: not yet. The number of rows is right, but some values differ."
        elif exp["ordered"] and _result_hash(rows, True) != exp["h"]:
            ok, reason = False, "order"
            text = f"Exercise {n}: almost. The rows are right, but they are not in the required order."
        else:
            ok, reason, text = True, "ok", f"Exercise {n}: correct!"
    _msg(("\u2714 " if ok else "\u2718 ") + text, "info" if ok else "warn")
    _record({"header": header, "check": n, "ok": ok, "reason": reason,
             "notebook": _os.environ.get("SQLNB_NOTEBOOK", "")})


def _run_sql(text, expect_error=False, check=None):
    """Run the SQL of one cell (the body of %%pg); with check=N, compare the last result with exercise N."""
    conn = _require_conn()
    stmts = _split_sql(text)
    header = _header(text)
    multi = sum(1 for s in stmts if (_strip_comments(s).split() or [""])[0].upper() in _ROW_KEYWORDS) > 1
    statuses, error, err_index, last = [], None, 0, None
    _ST["notices"].clear()

    def flush_status():
        if statuses:
            _msg("   ".join(statuses), "muted")
            statuses.clear()

    for idx, stmt in enumerate(stmts, 1):
        entry = {"header": header, "stmt_index": idx, "sql": stmt, "expect_error": expect_error,
                 "notebook": _os.environ.get("SQLNB_NOTEBOOK", "")}
        try:
            cur = conn.cursor()
            try:
                cur.execute(stmt)
            except psycopg.OperationalError:
                # a dead connection (runtime restarted, laptop slept) shows up on the first statement:
                # nothing ran on the server, so reconnect and retry once
                if idx != 1 or not (conn.broken or conn.closed):
                    raise
                conn = _reconnect()
                cur = conn.cursor()
                cur.execute(stmt)
            if cur.description is not None:
                cols = [d.name for d in cur.description]
                rows = cur.fetchall()
                flush_status()
                _flush_notices()
                _show_result(cols, rows, _caption(stmt) if multi else None)
                last = (cols, rows)
                entry.update(columns=cols, rows=[[_jsonable(v) for v in r] for r in rows], status=cur.statusmessage)
            else:
                statuses.append(cur.statusmessage or "OK")
                if _ST["notices"]:
                    flush_status()
                    _flush_notices()
                entry.update(columns=None, rows=None, status=cur.statusmessage or "OK")
            cur.close()
            _record(entry)
        except psycopg.Error as e:
            error, err_index = e, idx
            entry.update(error={"sqlstate": e.sqlstate, "message": e.diag.message_primary or str(e)})
            _record(entry)
            break
    flush_status()
    _flush_notices()

    if error is not None:
        rolled_back = False
        if conn.info.transaction_status == TransactionStatus.INERROR:
            conn.execute("ROLLBACK")
            rolled_back = True
        d = error.diag
        lines = [f"{'EXPECTED ERROR' if expect_error else 'ERROR'} [{error.sqlstate or '-'}]: "
                 f"{d.message_primary or str(error).strip()}"]
        if d.message_detail:
            lines.append(f"Detail: {d.message_detail}")
        if d.message_hint:
            lines.append(f"Hint: {d.message_hint}")
        if len(stmts) > 1:
            lines.append(f"(Statement {err_index} of {len(stmts)} in this cell; the remaining statements were not run.)")
        if rolled_back:
            lines.append("The open transaction was rolled back because of the error (ROLLBACK).")
        _msg("\n".join(lines), "expected" if expect_error else "error")
        if not expect_error and _os.environ.get("SQLNB_STRICT") == "1":
            raise SQLCellError(f"[{header}] {error.sqlstate}: {d.message_primary}")
    elif expect_error:
        _msg("WARNING: this cell was expected to fail, but every statement succeeded.", "warn")
        if _os.environ.get("SQLNB_STRICT") == "1":
            raise SQLCellError(f"[{header}] an error was expected but did not happen")
    if check is not None:
        _check(check, last, error, header)
    if conn.info.transaction_status == TransactionStatus.INTRANS:
        _msg("Note: a transaction is still open (BEGIN). Run COMMIT or ROLLBACK to finish it.", "warn")


def _show(*tables):
    """Print the current contents of tables (default: the six COMPANY tables)."""
    conn = _require_conn()
    for name in [t for t in tables if t] or _COMPANY_TABLES:
        if not _re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*)?", name):
            _msg(f"Invalid table name: {name}", "error")
            continue
        try:
            cur = conn.execute(f"SELECT * FROM {name}")
            cols, rows = [d.name for d in cur.description], cur.fetchall()
        except psycopg.Error as e:
            if conn.info.transaction_status == TransactionStatus.INERROR:
                conn.execute("ROLLBACK")
            _msg(f"{name}: {e.diag.message_primary or e}", "error")
            if _os.environ.get("SQLNB_STRICT") == "1":
                raise SQLCellError(f"%show {name}: {e}")
            continue
        n = len(rows)
        _display(_Out(f"{name.upper()} ({n} rows)",
                      f'<div style="font-weight:bold;margin-top:8px">{_html.escape(name.upper())} '
                      f'<span style="font-weight:normal;color:#888">({n} row{"" if n == 1 else "s"})</span></div>'))
        if rows:
            _display(_Out(_table_text(cols, rows), _table_html(cols, rows)))
        _record({"header": f"%show {name}", "stmt_index": 1, "sql": f"SELECT * FROM {name}", "columns": cols,
                 "rows": [[_jsonable(v) for v in r] for r in rows], "expect_error": False,
                 "notebook": _os.environ.get("SQLNB_NOTEBOOK", "")})


# ---------------------------------------------------------------- COMPANY schema and data (embedded)
_RESET_PRE = """
DROP SCHEMA IF EXISTS company CASCADE;
DROP SCHEMA IF EXISTS demo CASCADE;
DO $$
BEGIN
  IF EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'student') THEN
    EXECUTE 'DROP OWNED BY student';
    EXECUTE 'DROP ROLE student';
  END IF;
  IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'jsmith') THEN
    EXECUTE 'CREATE ROLE jsmith';
  END IF;
END $$;
CREATE SCHEMA company AUTHORIZATION jsmith;
CREATE SCHEMA demo;
"""

# Figure 6.1 + Figure 6.2 (identical to the CREATE TABLE cells of the Chapter 6 notebook)
_SCHEMA_SQL = """
-- EMPLOYEE first, without EMPDEPTFK (DEPARTMENT does not exist yet)
CREATE TABLE EMPLOYEE
( Fname           VARCHAR(15)     NOT NULL,
  Minit           CHAR,
  Lname           VARCHAR(15)     NOT NULL,
  Ssn             CHAR(9)         NOT NULL,
  Bdate           DATE,
  Address         VARCHAR(30),
  Sex             CHAR,
  Salary          DECIMAL(10,2),
  Super_ssn       CHAR(9),
  Dno             INT             NOT NULL    DEFAULT 1,
  CONSTRAINT EMPPK
    PRIMARY KEY (Ssn),
  CONSTRAINT EMPSUPERFK
    FOREIGN KEY (Super_ssn) REFERENCES EMPLOYEE(Ssn)
                ON DELETE SET NULL      ON UPDATE CASCADE );

CREATE TABLE DEPARTMENT
( Dname           VARCHAR(15)     NOT NULL,
  Dnumber         INT             NOT NULL,
  Mgr_ssn         CHAR(9)         NOT NULL    DEFAULT '888665555',
  Mgr_start_date  DATE,
  CONSTRAINT DEPTPK
    PRIMARY KEY (Dnumber),
  CONSTRAINT DEPTSK
    UNIQUE (Dname),
  CONSTRAINT DEPTMGRFK
    FOREIGN KEY (Mgr_ssn) REFERENCES EMPLOYEE(Ssn)
                ON DELETE SET DEFAULT   ON UPDATE CASCADE );

-- Now that DEPARTMENT exists, close the cycle (DEFERRABLE: see the data load below)
ALTER TABLE EMPLOYEE
  ADD CONSTRAINT EMPDEPTFK
    FOREIGN KEY (Dno) REFERENCES DEPARTMENT(Dnumber)
                ON DELETE SET DEFAULT   ON UPDATE CASCADE
    DEFERRABLE INITIALLY IMMEDIATE;

CREATE TABLE DEPT_LOCATIONS
( Dnumber         INT             NOT NULL,
  Dlocation       VARCHAR(15)     NOT NULL,
  PRIMARY KEY (Dnumber, Dlocation),
  FOREIGN KEY (Dnumber) REFERENCES DEPARTMENT(Dnumber)
                ON DELETE CASCADE       ON UPDATE CASCADE );

-- PROJECT, WORKS_ON and DEPENDENT are not shown in Figure 6.2: same as Figure 6.1
CREATE TABLE PROJECT
( Pname           VARCHAR(15)     NOT NULL,
  Pnumber         INT             NOT NULL,
  Plocation       VARCHAR(15),
  Dnum            INT             NOT NULL,
  PRIMARY KEY (Pnumber),
  UNIQUE (Pname),
  FOREIGN KEY (Dnum) REFERENCES DEPARTMENT(Dnumber) );

CREATE TABLE WORKS_ON
( Essn            CHAR(9)         NOT NULL,
  Pno             INT             NOT NULL,
  Hours           DECIMAL(3,1),                -- Figure 6.1 says NOT NULL; Figure 5.6 has a NULL
  PRIMARY KEY (Essn, Pno),
  FOREIGN KEY (Essn) REFERENCES EMPLOYEE(Ssn),
  FOREIGN KEY (Pno) REFERENCES PROJECT(Pnumber) );

CREATE TABLE DEPENDENT
( Essn            CHAR(9)         NOT NULL,
  Dependent_name  VARCHAR(15)     NOT NULL,
  Sex             CHAR,
  Bdate           DATE,
  Relationship    VARCHAR(8),
  PRIMARY KEY (Essn, Dependent_name),
  FOREIGN KEY (Essn) REFERENCES EMPLOYEE(Ssn) );
"""

# Figure 5.6 database state (slides 6-12, 6-13)
_DATA_SQL = """
-- EMPLOYEE and DEPARTMENT reference each other: load both in one transaction and
-- check EMPDEPTFK only at COMMIT
BEGIN;
SET CONSTRAINTS EMPDEPTFK DEFERRED;
INSERT INTO EMPLOYEE VALUES
  ('John',     'B', 'Smith',   '123456789', '1965-01-09', '731 Fondren, Houston, TX',  'M', 30000, '333445555', 5),
  ('Franklin', 'T', 'Wong',    '333445555', '1955-12-08', '638 Voss, Houston, TX',     'M', 40000, '888665555', 5),
  ('Alicia',   'J', 'Zelaya',  '999887777', '1968-01-19', '3321 Castle, Spring, TX',   'F', 25000, '987654321', 4),
  ('Jennifer', 'S', 'Wallace', '987654321', '1941-06-20', '291 Berry, Bellaire, TX',   'F', 43000, '888665555', 4),
  ('Ramesh',   'K', 'Narayan', '666884444', '1962-09-15', '975 Fire Oak, Humble, TX',  'M', 38000, '333445555', 5),
  ('Joyce',    'A', 'English', '453453453', '1972-07-31', '5631 Rice, Houston, TX',    'F', 25000, '333445555', 5),
  ('Ahmad',    'V', 'Jabbar',  '987987987', '1969-03-29', '980 Dallas, Houston, TX',   'M', 25000, '987654321', 4),
  ('James',    'E', 'Borg',    '888665555', '1937-11-10', '450 Stone, Houston, TX',    'M', 55000, NULL,        1);
INSERT INTO DEPARTMENT VALUES
  ('Research',       5, '333445555', '1988-05-22'),
  ('Administration', 4, '987654321', '1995-01-01'),
  ('Headquarters',   1, '888665555', '1981-06-19');
COMMIT;

INSERT INTO DEPT_LOCATIONS VALUES
  (1, 'Houston'), (4, 'Stafford'), (5, 'Bellaire'), (5, 'Sugarland'), (5, 'Houston');

INSERT INTO PROJECT VALUES
  ('ProductX',         1, 'Bellaire',  5),
  ('ProductY',         2, 'Sugarland', 5),
  ('ProductZ',         3, 'Houston',   5),
  ('Computerization', 10, 'Stafford',  4),
  ('Reorganization',  20, 'Houston',   1),
  ('Newbenefits',     30, 'Stafford',  4);

INSERT INTO WORKS_ON VALUES
  ('123456789',  1, 32.5), ('123456789',  2,  7.5),
  ('666884444',  3, 40.0),
  ('453453453',  1, 20.0), ('453453453',  2, 20.0),
  ('333445555',  2, 10.0), ('333445555',  3, 10.0), ('333445555', 10, 10.0), ('333445555', 20, 10.0),
  ('999887777', 30, 30.0), ('999887777', 10, 10.0),
  ('987987987', 10, 35.0), ('987987987', 30,  5.0),
  ('987654321', 30, 20.0), ('987654321', 20, 15.0),
  ('888665555', 20, NULL);

INSERT INTO DEPENDENT VALUES
  ('333445555', 'Alice',     'F', '1986-04-05', 'Daughter'),
  ('333445555', 'Theodore',  'M', '1983-10-25', 'Son'),
  ('333445555', 'Joy',       'F', '1958-05-03', 'Spouse'),
  ('987654321', 'Abner',     'M', '1942-02-28', 'Spouse'),
  ('123456789', 'Michael',   'M', '1988-01-04', 'Son'),
  ('123456789', 'Alice',     'F', '1988-12-30', 'Daughter'),
  ('123456789', 'Elizabeth', 'F', '1967-05-05', 'Spouse');
"""


def _reset_company(quiet=False):
    """Restore COMPANY to Figure 6.1/6.2 and the Figure 5.6 database state."""
    conn = _require_conn()
    if conn.info.transaction_status != TransactionStatus.IDLE:
        conn.execute("ROLLBACK")
    try:
        conn.execute("RESET ROLE")
    except psycopg.Error:
        pass
    try:
        for part in (_RESET_PRE, _SCHEMA_SQL, _DATA_SQL):
            for stmt in _split_sql(part):
                conn.execute(stmt)
    except psycopg.Error:
        if conn.info.transaction_status == TransactionStatus.INERROR:
            conn.execute("ROLLBACK")
        raise
    finally:
        _ST["notices"].clear()
    counts = conn.execute(
        "SELECT (SELECT COUNT(*) FROM employee), (SELECT COUNT(*) FROM department), (SELECT COUNT(*) FROM dept_locations),"
        " (SELECT COUNT(*) FROM project), (SELECT COUNT(*) FROM works_on), (SELECT COUNT(*) FROM dependent)").fetchone()
    if not quiet:
        _msg("COMPANY restored to Figure 6.1/6.2 + Figure 5.6: " +
             ", ".join(f"{t}={c}" for t, c in zip(_COMPANY_TABLES, counts)))


# ---------------------------------------------------------------- notebook commands
def _parse_flags(line):
    expect, check, rest = False, None, (line or "").strip()
    while True:
        m = _re.match(r"^(--expect-error|-e)(\s+|$)", rest)
        if m:
            expect, rest = True, rest[m.end():]
            continue
        m = _re.match(r"^--check\s+(\d+)(\s+|$)", rest)
        if m:
            check, rest = int(m.group(1)), rest[m.end():]
            continue
        return expect, check, rest


def _pg_magic(line, cell=None):
    expect, check, rest = _parse_flags(line)
    text = cell if cell is not None else rest
    if not text or not text.strip():
        _msg("No SQL to run.", "warn")
        return
    try:
        _run_sql(text, expect_error=expect, check=check)
    except SetupError as e:
        _msg(str(e), "error")


def _show_magic(line):
    try:
        _show(*(line or "").replace(",", " ").split())
    except SetupError as e:
        _msg(str(e), "error")


def _register_magics():
    try:
        from IPython import get_ipython
    except ImportError:
        return
    ip = get_ipython()
    if ip is not None:
        ip.register_magic_function(_pg_magic, magic_kind="line_cell", magic_name="pg")
        ip.register_magic_function(_show_magic, magic_kind="line", magic_name="show")


sqlnb = _types.SimpleNamespace(setup=_setup, reset_company=_reset_company, show=_show, run_sql=_run_sql,
                               stop=_stop, remove_container=_remove_container, connection_info=_connection_info,
                               connect=_connect, split_sql=_split_sql)
_msg("Helper loaded. Next: run the setup cell below (sqlnb.setup()).", "muted")

In [ ]:
# Start PostgreSQL (Colab: installed with apt-get; on your computer: Docker) and connect
sqlnb.setup(reset=True)

**The COMPANY schema** (primary keys in bold; → is a foreign key). Run `%show` to see all the rows.

| Table | Columns |
|---|---|
| EMPLOYEE | Fname, Minit, Lname, **Ssn**, Bdate, Address, Sex, Salary, Super_ssn → EMPLOYEE, Dno → DEPARTMENT |
| DEPARTMENT | Dname, **Dnumber**, Mgr_ssn → EMPLOYEE, Mgr_start_date |
| DEPT_LOCATIONS | **Dnumber** → DEPARTMENT, **Dlocation** |
| PROJECT | Pname, **Pnumber**, Plocation, Dnum → DEPARTMENT |
| WORKS_ON | **Essn** → EMPLOYEE, **Pno** → PROJECT, Hours |
| DEPENDENT | **Essn** → EMPLOYEE, **Dependent_name**, Sex, Bdate, Relationship |

In [ ]:
# Slides 6-12, 6-13 | The Figure 5.6 database state
%show

### Exercise 1  ★☆☆  Selection and ordering
*Easy · slides 6-31, 6-42*

**Query:** List the first name, last name and salary of every **female** employee, ordered by salary from highest to lowest. Employees with the same salary are listed in alphabetical order of last name.

**Result columns (in this order):** Fname, Lname, Salary · **the order matters**

<details><summary><b>Hint</b> (click to open)</summary>

Filter with `WHERE`, sort with `ORDER BY`. `ORDER BY` can list several columns, each with its own `ASC`/`DESC`.

</details>

In [ ]:
%%pg --check 1
-- Exercise 1 | Your answer
-- Write your query below this line and run the cell.


### Exercise 2  ★☆☆  Joining two tables
*Easy · slides 6-31, 7-19, 7-21*

**Query:** For every project located in **Houston**, list the project name and the name of the department that controls the project.

**Result columns (in this order):** Pname, Dname

<details><summary><b>Hint</b> (click to open)</summary>

`PROJECT.Dnum` refers to `DEPARTMENT.Dnumber`.

</details>

In [ ]:
%%pg --check 2
-- Exercise 2 | Your answer
-- Write your query below this line and run the cell.


### Exercise 3  ★☆☆  LIKE and BETWEEN
*Easy · slides 6-39, 6-40*

**Query:** List the first name, last name and salary of every employee who **lives in Houston** and whose salary is **between 25000 and 35000** (both limits included).

**Result columns (in this order):** Fname, Lname, Salary

<details><summary><b>Hint</b> (click to open)</summary>

Look closely at how the addresses are written, e.g. `'731 Fondren, Houston, TX'` (slide 6-40). `BETWEEN` includes both limits.

</details>

In [ ]:
%%pg --check 3
-- Exercise 3 | Your answer
-- Write your query below this line and run the cell.


### Exercise 4  ★★☆  Self-join
*Medium · slides 6-34, 6-35*

**Query:** For every employee who works in a **different department from his or her direct supervisor**, list the employee's last name and department number, followed by the supervisor's last name and department number.

**Result columns (in this order):** employee Lname, employee Dno, supervisor Lname, supervisor Dno

<details><summary><b>Hint</b> (click to open)</summary>

Use `EMPLOYEE` twice with two aliases, as in Q8 (slide 6-34): one for the employee, one for the supervisor.

</details>

In [ ]:
%%pg --check 4
-- Exercise 4 | Your answer
-- Write your query below this line and run the cell.


### Exercise 5  ★★☆  GROUP BY and HAVING
*Medium · slides 7-29 to 7-33*

**Query:** For every project on which the employees work **more than 40 hours per week in total**, list the project name, the number of employees who work on it, and the total weekly hours. Order the result by total hours (largest first); projects with the same total are ordered by project name.

**Result columns (in this order):** Pname, number of employees, total hours · **the order matters**

<details><summary><b>Hint</b> (click to open)</summary>

A condition on a group (a total) belongs in `HAVING`, not in `WHERE` (slides 7-31 to 7-33).

</details>

In [ ]:
%%pg --check 5
-- Exercise 5 | Your answer
-- Write your query below this line and run the cell.


### Exercise 6  ★★☆  Outer join and COUNT
*Medium · slides 7-22, 7-27, 7-30*

**Query:** List the first name and last name of **every** employee together with the number of dependents he or she has; employees without dependents must appear with **0**. Order the result by the number of dependents (largest first), then by last name.

**Result columns (in this order):** Fname, Lname, number of dependents · **the order matters**

<details><summary><b>Hint</b> (click to open)</summary>

Which join keeps the employees that have no matching `DEPENDENT` row? And what does `COUNT(*)` count for such an employee?

</details>

In [ ]:
%%pg --check 6
-- Exercise 6 | Your answer
-- Write your query below this line and run the cell.


### Exercise 7  ★★☆  Correlated nested query (EXISTS)
*Medium · slides 7-12 to 7-15*

**Query:** List the first and last names of the employees who work on **at least one project that is not controlled by their own department**. Each employee appears once.

**Result columns (in this order):** Fname, Lname

<details><summary><b>Hint</b> (click to open)</summary>

Compare the project's controlling department (`PROJECT.Dnum`) with the employee's department (`EMPLOYEE.Dno`) inside a correlated `EXISTS` (slide 7-15).

</details>

In [ ]:
%%pg --check 7
-- Exercise 7 | Your answer
-- Write your query below this line and run the cell.


### Exercise 8  ★★★  Division: "every"
*Hard · slides 7-16, 7-17*

**Query:** List the first and last names of the employees who work on **every** project that John Smith works on. Do not include John Smith himself.

**Result columns (in this order):** Fname, Lname

<details><summary><b>Hint</b> (click to open)</summary>

"Every project of Smith" means: there is **no** project of Smith that the employee does **not** work on (double negation, slides 7-16 and 7-17).

</details>

In [ ]:
%%pg --check 8
-- Exercise 8 | Your answer
-- Write your query below this line and run the cell.


### Exercise 9  ★★★  The maximum within each group
*Hard · slides 7-11, 7-12, 7-34*

**Query:** For each department, list the department name and the first name, last name and salary of the employee(s) who earn the **highest salary in that department**.

**Result columns (in this order):** Dname, Fname, Lname, Salary

<details><summary><b>Hint</b> (click to open)</summary>

Compare each employee's salary with the maximum salary of **his or her own** department: a correlated nested query (slide 7-12) or a `WITH` clause (slide 7-34).

</details>

In [ ]:
%%pg --check 9
-- Exercise 9 | Your answer
-- Write your query below this line and run the cell.


### Exercise 10  ★★★  Recursive query
*Hard · slides 7-38, 7-39*

**Query:** List the last names of all employees who are supervised by James Borg **directly or indirectly**, together with their level below Borg: 1 for employees whom Borg supervises directly, 2 for employees supervised by someone at level 1, and so on.

**Result columns (in this order):** Lname, level

<details><summary><b>Hint</b> (click to open)</summary>

`WITH RECURSIVE` (slide 7-39): start with Borg's direct supervisees at level 1; each step adds the supervisees of the rows found so far, one level deeper.

</details>

In [ ]:
%%pg --check 10
-- Exercise 10 | Your answer
-- Write your query below this line and run the cell.
